# 🏆 Day 20 — GRAND FINALE: Model Serving at Scale, Advanced Monitoring & Final Capstone
**Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | Model Serving at Scale: Async Batching & Parallelism | ~11 sec |
| 2 | 10:30–11:00 AM | Advanced Drift Monitoring | ~5 sec |
| 3 | 11:00 AM–1:00 PM | Cost Optimisation: Quantisation, Caching & ROI | ~20 sec |
| 4 | 1:00–2:00 PM | Lunch Break & 20-Day Reflection | — |
| 5 | 2:00–4:00 PM | 20-Day Grand Finale: Complete Production ML System | ~20 sec |

> **Zero downloads. Pure numpy + sklearn + optuna + scipy.**
---

In [ ]:
# !pip install scikit-learn optuna scipy matplotlib pandas numpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance, ks_2samp
from scipy import stats
import warnings, time, hashlib, json
from datetime import datetime
from collections import deque
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, load_wine, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, brier_score_loss,
    confusion_matrix
)
from sklearn.isotonic import IsotonicRegression
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import sklearn
print('All imports OK')
print(f'   sklearn : {sklearn.__version__}')
print(f'   optuna  : {optuna.__version__}')

---
## Session 1 — Model Serving at Scale
**Time:** 9:00–10:30 AM | **Run time: ~11 sec**

In [ ]:
# 1.1  Async Dynamic Batch Server
class AsyncBatchServer:
    """
    Simulates async dynamic batching:
    - Requests are enqueued with arrival timestamps
    - Flush when max_batch_size reached OR max_wait_ms elapsed
    - Enables high GPU utilisation with bounded latency
    """
    def __init__(self, model_fn, max_batch=32, max_wait_ms=10):
        self.model_fn   = model_fn
        self.queue      = deque()
        self.max_batch  = max_batch
        self.max_wait   = max_wait_ms / 1000.0
        self.n_batches  = 0
        self.n_requests = 0
        self.latencies  = []

    def enqueue(self, x):
        self.queue.append({'x': x, 'arrival': time.perf_counter()})
        self.n_requests += 1

    def flush(self):
        """Process a batch. Returns predictions."""
        batch = []
        while self.queue and len(batch) < self.max_batch:
            item = self.queue.popleft()
            age  = time.perf_counter() - item['arrival']
            if age > self.max_wait * 10:   # discard stale (demo only)
                continue
            batch.append(item['x'])
        if not batch:
            return None
        t0       = time.perf_counter()
        X_batch  = np.vstack(batch)
        result   = self.model_fn(X_batch)
        latency  = (time.perf_counter() - t0) * 1000
        self.latencies.append(latency)
        self.n_batches += 1
        return result

    def stats(self):
        lats = np.array(self.latencies)
        return {
            'total_requests' : self.n_requests,
            'total_batches'  : self.n_batches,
            'avg_batch_size' : self.n_requests / (self.n_batches + 1e-9),
            'p50_latency_ms' : np.percentile(lats, 50) if len(lats)>0 else 0,
            'p99_latency_ms' : np.percentile(lats, 99) if len(lats)>0 else 0,
        }

# Setup model
X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)
rf_serve = RandomForestClassifier(80, random_state=42).fit(X_tr, y_tr)

# Benchmark: single vs batched serving
N_REQUESTS = 200

# Single-sample serving
t0 = time.perf_counter()
for i in range(N_REQUESTS):
    rf_serve.predict_proba(X_te[[i % len(X_te)]])
single_time = (time.perf_counter() - t0) * 1000

# Batch serving
server = AsyncBatchServer(rf_serve.predict_proba, max_batch=32, max_wait_ms=5)
for i in range(N_REQUESTS):
    server.enqueue(X_te[[i % len(X_te)]])

batch_time = 0
while server.queue:
    t0 = time.perf_counter()
    server.flush()
    batch_time += (time.perf_counter() - t0) * 1000

print(f'Single-sample serving : {single_time:.1f}ms  ({N_REQUESTS} requests)')
print(f'Batch serving         : {batch_time:.1f}ms  ({N_REQUESTS} requests)')
print(f'Speedup               : {single_time/batch_time:.2f}x')
st = server.stats()
print(f'Avg batch size        : {st["avg_batch_size"]:.1f}')

In [ ]:
# 1.2  Priority Queue Serving
import heapq

class PriorityBatchServer:
    """
    Priority-based serving: higher priority requests
    jump the queue (e.g. premium tier gets lower latency).
    """
    def __init__(self, model_fn, max_batch=16):
        self.model_fn = model_fn
        self.heap     = []  # min-heap on (priority_score, counter, x)
        self.counter  = 0
        self.tier_stats = {'premium': 0, 'standard': 0}

    def enqueue(self, x, priority=0):  # lower = higher priority
        heapq.heappush(self.heap, (priority, self.counter, x))
        self.counter += 1
        tier = 'premium' if priority < 5 else 'standard'
        self.tier_stats[tier] += 1

    def flush(self, k=None):
        k = k or min(len(self.heap), 16)
        batch = []
        for _ in range(k):
            if not self.heap: break
            _, _, x = heapq.heappop(self.heap)
            batch.append(x)
        if not batch: return None
        return self.model_fn(np.vstack(batch))

# Demo: mix of premium and standard requests
ps = PriorityBatchServer(rf_serve.predict_proba)
rng_ps = np.random.default_rng(42)
for i in range(30):
    tier     = rng_ps.choice(['premium', 'standard'], p=[0.3, 0.7])
    priority = rng_ps.integers(0, 4) if tier=='premium' else rng_ps.integers(5, 10)
    ps.enqueue(X_te[[i % len(X_te)]], priority=priority)

result_ps = ps.flush(k=16)
print(f'Priority queue demo:')
print(f'  Total enqueued : {ps.counter}')
print(f'  Premium        : {ps.tier_stats["premium"]}')
print(f'  Standard       : {ps.tier_stats["standard"]}')
print(f'  Batch result   : {result_ps.shape}')

In [ ]:
# 1.3  Throughput vs Latency Curve
batch_sizes  = [1, 2, 4, 8, 16, 32, 64]
throughputs  = []
latencies_bs = []

for bs in batch_sizes:
    X_batch = X_te[:bs]
    times   = []
    for _ in range(50):     # 50 warmup+measure iterations
        t0 = time.perf_counter()
        rf_serve.predict_proba(X_batch)
        times.append((time.perf_counter() - t0) * 1000)
    avg_lat  = np.mean(times)
    throughput = bs / (avg_lat / 1000)  # samples per second
    latencies_bs.append(avg_lat)
    throughputs.append(throughput)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogx(batch_sizes, latencies_bs, 'o-', color='#D85A30', linewidth=2, markersize=7)
axes[0].set_xlabel('Batch size'); axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Latency vs Batch Size')
axes[0].grid(alpha=0.3)

axes[1].semilogx(batch_sizes, throughputs, 's-', color='#534AB7', linewidth=2, markersize=7)
axes[1].set_xlabel('Batch size'); axes[1].set_ylabel('Throughput (samples/sec)')
axes[1].set_title('Throughput vs Batch Size')
axes[1].grid(alpha=0.3)

plt.suptitle('Serving Trade-Off: Latency vs Throughput', fontsize=12)
plt.tight_layout(); plt.show()

print('Batch size | Latency (ms) | Throughput (samples/sec)')
for bs, lat, thr in zip(batch_sizes, latencies_bs, throughputs):
    print(f'  {bs:10d} | {lat:12.3f} | {thr:,.0f}')

---
## Session 2 — Advanced Drift Monitoring
**Time:** 10:30–11:00 AM | **Run time: ~5 sec**

In [ ]:
# 2.1  Drift Detection: PSI vs Wasserstein vs KS vs MMD
rng = np.random.default_rng(42)
N   = 1000

scenarios = {
    'No drift'     : rng.normal(0,   1.0, N),
    'Mild drift'   : rng.normal(0.3, 1.1, N),
    'Moderate'     : rng.normal(0.7, 1.3, N),
    'Severe drift' : rng.normal(1.5, 1.8, N),
}
reference = rng.normal(0, 1.0, N)

def psi(expected, actual, n_bins=10):
    bins = np.percentile(expected, np.linspace(0, 100, n_bins+1))
    bins = np.unique(bins)
    e = np.histogram(expected, bins=bins)[0] / len(expected) + 1e-8
    a = np.histogram(actual,   bins=bins)[0] / len(actual)   + 1e-8
    return float(np.sum((a - e) * np.log(a / e)))

def mmd_rbf(X, Y, gamma=1.0):
    """Maximum Mean Discrepancy with RBF kernel."""
    def rbf(a, b):
        diff = a[:, None] - b[None, :]
        return np.exp(-gamma * diff**2)
    Kxx = rbf(X, X); Kyy = rbf(Y, Y); Kxy = rbf(X, Y)
    n, m = len(X), len(Y)
    return float(Kxx.mean() + Kyy.mean() - 2 * Kxy.mean())

print(f'{"Scenario":15s} | {"PSI":6s} | {"Wasserstein":12s} | {"KS stat":8s} | {"MMD":6s} | Alert')
print('-' * 72)
for name, prod in scenarios.items():
    psi_v  = psi(reference, prod)
    wd     = wasserstein_distance(reference, prod)
    ks_s   = ks_2samp(reference, prod).statistic
    # MMD on subsample for speed
    mmd_v  = mmd_rbf(reference[:200], prod[:200])
    level  = 'NONE' if psi_v<0.1 else 'WARN' if psi_v<0.2 else 'CRITICAL'
    print(f'{name:15s} | {psi_v:6.4f} | {wd:12.4f} | {ks_s:8.4f} | {mmd_v:6.4f} | {level}')

In [ ]:
# 2.2  Feature-Level Drift Dashboard
rng = np.random.default_rng(7)
N_FEAT = X_bc.shape[1]
train_feats = X_bc[:400]   # simulated training distribution
prod_feats  = X_bc[400:] + rng.normal(0, 0.3, (len(X_bc)-400, N_FEAT))

feature_names = load_breast_cancer().feature_names
drift_scores  = []
for i in range(N_FEAT):
    wd = wasserstein_distance(train_feats[:, i], prod_feats[:, i])
    drift_scores.append(wd)

drift_df = pd.DataFrame({
    'feature': feature_names,
    'drift'  : drift_scores
}).sort_values('drift', ascending=False)

print('Top 10 Drifting Features:')
print(drift_df.head(10).to_string(index=False))

# Alert: features exceeding threshold
WARN_THRESH  = 0.5
ALERT_THRESH = 1.0
alert_feats  = drift_df[drift_df['drift'] > ALERT_THRESH]
warn_feats   = drift_df[(drift_df['drift'] > WARN_THRESH) & (drift_df['drift'] <= ALERT_THRESH)]
print(f'\nALERT: {len(alert_feats)} features | WARN: {len(warn_feats)} features')

In [ ]:
# 2.3  Concept Drift: Label Relationship Change
rng = np.random.default_rng(0)

# Simulate: same features, but label relationship changes over time
def simulate_concept_drift(n_windows=6, n_per_window=200):
    results = []
    for w in range(n_windows):
        X_w = rng.normal(0, 1, (n_per_window, 5))
        # Label flips progressively as concept drifts
        drift_strength = w * 0.15
        y_w = ((X_w[:, 0] - drift_strength*X_w[:,1] +
                rng.normal(0, 0.5, n_per_window)) > 0).astype(int)
        results.append((X_w, y_w))
    return results

windows = simulate_concept_drift()
# Train on window 0, evaluate on all windows
m_concept = LogisticRegression(max_iter=300)
m_concept.fit(windows[0][0], windows[0][1])

print('Concept drift — model accuracy across time windows:')
for w, (X_w, y_w) in enumerate(windows):
    acc = m_concept.score(X_w, y_w)
    bar = '█' * int(acc * 20)
    drift_flag = ' ← CONCEPT DRIFT' if acc < 0.6 else ''
    print(f'  Window {w}: {acc:.4f}  {bar}{drift_flag}')

---
## Session 3 — Cost Optimisation
**Time:** 11:00 AM–1:00 PM | **Run time: ~20 sec**

In [ ]:
# 3.1  Semantic Prediction Cache
class SemanticCache:
    """
    Cache predictions by MD5 hash of input.
    High hit rate when requests have many repeated inputs.
    """
    def __init__(self, model_fn, max_size=10_000):
        self.model_fn = model_fn
        self.cache    = {}
        self.max_size = max_size
        self.hits     = 0
        self.misses   = 0
        self.t_cache  = 0.0
        self.t_model  = 0.0

    def predict(self, x):
        key = hashlib.md5(x.tobytes()).hexdigest()
        t0  = time.perf_counter()
        if key in self.cache:
            self.hits += 1
            self.t_cache += time.perf_counter() - t0
            return self.cache[key]
        # Cache miss — run model
        t1     = time.perf_counter()
        result = self.model_fn(x.reshape(1,-1))[0]
        self.t_model += time.perf_counter() - t1
        self.misses  += 1
        if len(self.cache) >= self.max_size:
            # LRU eviction: remove oldest (simplified)
            self.cache.pop(next(iter(self.cache)))
        self.cache[key] = result
        return result

    @property
    def hit_rate(self):
        return self.hits / (self.hits + self.misses + 1e-9)

    def cost_report(self, cost_per_inference=0.001):
        """Cost saved from cache hits."""
        saved = self.hits * cost_per_inference
        total_without = (self.hits + self.misses) * cost_per_inference
        return {'hit_rate': self.hit_rate,
                'cost_saved': saved,
                'total_without_cache': total_without,
                'cost_reduction_pct': saved / (total_without + 1e-9)}

# Simulate realistic traffic: 80/20 rule (20% unique, 80% repeated)
cache = SemanticCache(rf_serve.predict_proba, max_size=1000)
rng_cache = np.random.default_rng(42)
unique_pool = X_te[:20]   # only 20 unique queries

for _ in range(1000):
    # 80% of requests are one of the first 5 (very popular)
    if rng_cache.random() < 0.8:
        idx = rng_cache.integers(0, 5)
    else:
        idx = rng_cache.integers(0, 20)
    cache.predict(unique_pool[idx])

report = cache.cost_report(cost_per_inference=0.001)  # $0.001 per inference
print(f'Semantic Cache Report:')
print(f'  Hit rate          : {report["hit_rate"]:.1%}')
print(f'  Cost without cache: ${report["total_without_cache"]:.3f}')
print(f'  Cost saved        : ${report["cost_saved"]:.3f}  ({report["cost_reduction_pct"]:.1%})')

In [ ]:
# 3.2  Quantisation Cost Savings Simulation
def simulate_quantisation_savings(model_size_mb, base_latency_ms, n_requests_per_day):
    """Compare float32 vs int8 serving costs."""
    GPU_PRICE_PER_HOUR = 3.0    # $3 per GPU-hour
    scenarios = {
        'float32' : {'size_factor':1.0, 'lat_factor':1.0, 'accuracy_drop':0.000},
        'float16' : {'size_factor':0.5, 'lat_factor':0.7, 'accuracy_drop':0.001},
        'int8'    : {'size_factor':0.25,'lat_factor':0.45,'accuracy_drop':0.005},
        'int4'    : {'size_factor':0.125,'lat_factor':0.30,'accuracy_drop':0.015},
    }
    print(f'Model size: {model_size_mb}MB  Base latency: {base_latency_ms}ms  '
          f'Traffic: {n_requests_per_day:,} req/day')
    print(f'{"Format":10s} | {"Size(MB)":9s} | {"Lat(ms)":8s} | {"GPU-hrs/day":11s} | {"Cost/day":9s} | {"Acc drop"}')
    print('-' * 72)
    for fmt, params in scenarios.items():
        size    = model_size_mb  * params['size_factor']
        lat     = base_latency_ms * params['lat_factor']
        gpu_hrs = (n_requests_per_day * lat / 1000) / 3600
        cost    = gpu_hrs * GPU_PRICE_PER_HOUR
        print(f'{fmt:10s} | {size:9.1f} | {lat:8.2f} | {gpu_hrs:11.2f} | '
              f'${cost:8.2f} | {params["accuracy_drop"]*100:.2f}%')

simulate_quantisation_savings(
    model_size_mb=500, base_latency_ms=20, n_requests_per_day=100_000
)

In [ ]:
# 3.3  Cost-Quality Pareto Frontier
rng = np.random.default_rng(42)
X_w, y_w = load_wine(return_X_y=True)

# Simulate cost in compute units (proportional to model complexity)
cost_models = []
for n_est in [5, 10, 20, 50, 100, 200]:
    for depth in [2, 4, 8]:
        m   = RandomForestClassifier(n_est, max_depth=depth, random_state=42)
        acc = cross_val_score(m, X_w, y_w, cv=3).mean()
        cost = n_est * depth  # proxy for inference FLOPs
        cost_models.append({'n_est': n_est, 'depth': depth,
                            'accuracy': acc, 'cost': cost})

df_cost = pd.DataFrame(cost_models)

# Pareto frontier: best accuracy for given cost
def pareto(df, quality_col, cost_col):
    df_s = df.sort_values(cost_col)
    best = -np.inf
    front = []
    for _, row in df_s.iterrows():
        if row[quality_col] > best:
            best = row[quality_col]
            front.append(row)
    return pd.DataFrame(front)

pareto_df = pareto(df_cost, 'accuracy', 'cost')

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df_cost['cost'], df_cost['accuracy'], alpha=0.4, color='#888780', s=40)
ax.plot(pareto_df['cost'], pareto_df['accuracy'], 'o-', color='#534AB7',
        linewidth=2, markersize=8, label='Pareto frontier')
ax.set_xlabel('Inference cost (proxy)')
ax.set_ylabel('CV Accuracy')
ax.set_title('Cost-Quality Pareto Frontier')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Pareto-optimal models: {len(pareto_df)}')
print(pareto_df[['n_est','depth','accuracy','cost']].to_string(index=False))

---
## Lunch Break & 20-Day Reflection — 1:00–2:00 PM

**Skills gained across 20 days:**

- Days 1–5 : NumPy → Pandas → sklearn → Ensembles → Deep Learning → Deployment
- Days 6–10: Stats → Recommenders → Fraud → LLMs → MLOps → Data Quality
- Days 11–15: Causal → Bandits → Federated → DP → Active Learning → Attention
- Days 16–20: MoE → Contrastive → Conformal → RLHF → RAG → Scale & Monitoring
---

## Session 5 — 20-Day Grand Finale: Complete Production ML System
**Time:** 2:00–4:00 PM | **Run time: ~20 sec**

In [ ]:
# GRAND FINALE: Every major concept from all 20 days in one pipeline
print('='*65)
print(' 20-DAY GRAND FINALE: COMPLETE PRODUCTION ML SYSTEM')
print('='*65)

X_fin, y_fin = load_breast_cancer(return_X_y=True)
rng_fin = np.random.default_rng(0)
group_fin = rng_fin.choice([0,1], len(y_fin), p=[0.55,0.45])  # protected attr

# Three-way split: train / calibration / test
X_tr_f, X_tmp_f, y_tr_f, y_tmp_f, g_tr_f, g_tmp_f = train_test_split(
    X_fin, y_fin, group_fin, test_size=0.4, random_state=42
)
X_cal_f, X_te_f, y_cal_f, y_te_f, g_cal_f, g_te_f = train_test_split(
    X_tmp_f, y_tmp_f, g_tmp_f, test_size=0.5, random_state=42
)

print(f'Dataset: {X_fin.shape}  Train:{len(X_tr_f)}  Cal:{len(X_cal_f)}  Test:{len(X_te_f)}')

In [ ]:
# Stage 1: Bayesian HPO (Day 14 skill)
print('\nStage 1 — Bayesian HPO (Optuna TPE)')

def objective_fin(trial):
    m = GradientBoostingClassifier(
        n_estimators  = trial.suggest_int('n', 50, 200),
        learning_rate = trial.suggest_float('lr', 0.01, 0.3, log=True),
        max_depth     = trial.suggest_int('d', 2, 6),
        subsample     = trial.suggest_float('ss', 0.6, 1.0),
        random_state  = 42
    )
    return cross_val_score(m, X_tr_f, y_tr_f, cv=3, scoring='roc_auc').mean()

study_fin = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
study_fin.optimize(objective_fin, n_trials=25, show_progress_bar=False)
bp = study_fin.best_params
print(f'  Best params: {bp}')
print(f'  Best CV AUC: {study_fin.best_value:.4f}')

In [ ]:
# Stage 2: Train + Calibrate (Day 17 skill)
print('\nStage 2 — Train & Calibrate')

model_fin = GradientBoostingClassifier(
    n_estimators=bp['n'], learning_rate=bp['lr'],
    max_depth=bp['d'], subsample=bp['ss'], random_state=42
).fit(X_tr_f, y_tr_f)

# Isotonic calibration
prob_cal = model_fin.predict_proba(X_cal_f)[:, 1]
iso_fin  = IsotonicRegression(out_of_bounds='clip').fit(prob_cal, y_cal_f)

prob_te_raw = model_fin.predict_proba(X_te_f)[:, 1]
prob_te_cal = iso_fin.predict(prob_te_raw)

# ECE before/after
def ece_metric(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    ece  = 0.0
    for i in range(n_bins):
        m = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if m.sum() == 0: continue
        ece += m.sum()/len(y_true) * abs(y_true[m].mean() - y_prob[m].mean())
    return ece

ece_before = ece_metric(y_te_f, prob_te_raw)
ece_after  = ece_metric(y_te_f, prob_te_cal)
auc_final  = roc_auc_score(y_te_f, prob_te_cal)
print(f'  AUC              : {auc_final:.4f}')
print(f'  ECE before cal   : {ece_before:.4f}')
print(f'  ECE after  cal   : {ece_after:.4f}')

In [ ]:
# Stage 3: Fairness Audit (Day 13 skill)
print('\nStage 3 — Fairness Audit')
pred_fin = (prob_te_cal >= 0.5).astype(int)

for g, name in [(0,'Group 0'),(1,'Group 1')]:
    m  = g_te_f == g
    tp = ((pred_fin[m]==1)&(y_te_f[m]==1)).sum()
    fn = ((pred_fin[m]==0)&(y_te_f[m]==1)).sum()
    fp = ((pred_fin[m]==1)&(y_te_f[m]==0)).sum()
    tn = ((pred_fin[m]==0)&(y_te_f[m]==0)).sum()
    tpr = tp/(tp+fn+1e-9); fpr = fp/(fp+tn+1e-9)
    pos = pred_fin[m].mean()
    print(f'  {name}: TPR={tpr:.4f}  FPR={fpr:.4f}  PosRate={pos:.4f}')

rates = [pred_fin[g_te_f==g].mean() for g in [0,1]]
di = min(rates)/max(rates)
print(f'  Disparate Impact ratio: {di:.4f}  (fair={di>=0.8})')

In [ ]:
# Stage 4: Conformal Prediction Wrapper (Day 16 skill)
print('\nStage 4 — Conformal Prediction')

nonconf = 1 - iso_fin.predict(model_fin.predict_proba(X_cal_f)[:, 1])
for alpha in [0.05, 0.10, 0.20]:
    q_hat = np.quantile(nonconf, 1-alpha)
    test_probs = model_fin.predict_proba(X_te_f)
    sets = [np.where((1-p) <= q_hat)[0] for p in test_probs]
    cov  = np.mean([y_te_f[i] in sets[i] for i in range(len(y_te_f))])
    avg_sz = np.mean([len(s) for s in sets])
    ok = '✓' if cov >= 1-alpha-0.03 else '✗'
    print(f'  α={alpha:.2f}: coverage={cov:.1%} {ok}  avg_set_size={avg_sz:.3f}')

In [ ]:
# Stage 5: Production Serving (Day 20 skill)
print('\nStage 5 — Production Serving')

# Async batch server
serv = AsyncBatchServer(model_fin.predict_proba, max_batch=32)
for i in range(100):
    serv.enqueue(X_te_f[[i % len(X_te_f)]])
while serv.queue:
    serv.flush()

# Semantic cache
cache_fin = SemanticCache(model_fin.predict_proba, max_size=500)
for i in range(300):
    cache_fin.predict(X_te_f[i % 20])

# Latency profiling
lats_fin = []
for i in range(200):
    t0 = time.perf_counter()
    model_fin.predict_proba(X_te_f[[i % len(X_te_f)]])
    lats_fin.append((time.perf_counter()-t0)*1000)
lats_arr = np.array(lats_fin)

print(f'  Async server batches: {serv.n_batches}')
print(f'  Cache hit rate      : {cache_fin.hit_rate:.1%}')
print(f'  Latency p50         : {np.percentile(lats_arr,50):.3f}ms')
print(f'  Latency p99         : {np.percentile(lats_arr,99):.3f}ms')

In [ ]:
# Stage 6: Drift Monitoring & Auto-Retrain (Day 8 + Day 20)
print('\nStage 6 — Drift Monitoring')

rng_mon = np.random.default_rng(42)
N_MON   = 90   # 90 days monitoring
base_auc = auc_final
results_mon = []

for day in range(1, N_MON+1):
    drift = day * 0.001 + rng_mon.normal(0, 0.002)
    curr_auc = max(0.75, base_auc - drift + rng_mon.normal(0, 0.003))
    # Wasserstein on simulated score distribution
    train_scores = model_fin.predict_proba(X_tr_f)[:, 1]
    prod_noise   = rng_mon.normal(drift, 0.05, len(train_scores))
    prod_scores  = np.clip(train_scores + prod_noise, 0, 1)
    wd           = wasserstein_distance(train_scores, prod_scores)
    retrain      = wd > 0.15 or (base_auc - curr_auc) > 0.02
    if retrain:
        base_auc = curr_auc + 0.01  # retrain partially recovers
    results_mon.append({'day':day,'auc':curr_auc,'wd':wd,'retrain':retrain})

df_mon = pd.DataFrame(results_mon)
retrain_days = df_mon[df_mon['retrain']]
print(f'  Monitoring period      : {N_MON} days')
print(f'  Retraining events      : {len(retrain_days)}')
print(f'  Final AUC              : {df_mon.auc.iloc[-1]:.4f}')
print(f'  Avg Wasserstein drift  : {df_mon.wd.mean():.4f}')

In [ ]:
# FINAL GRAND REPORT
print('\n' + '='*65)
print(' 20-DAY GRAND FINALE — FINAL SYSTEM REPORT')
print('='*65)
print(f' Date           : {datetime.utcnow().strftime("%Y-%m-%d")}')
print(f' Model          : GradientBoostingClassifier (HPO-tuned)')
print(f' Best HPO AUC   : {study_fin.best_value:.4f}')
print(f' Test AUC       : {auc_final:.4f}')
print(f' ECE (calibrated): {ece_after:.4f}')
print(f' DI ratio       : {di:.4f}  (fair={di>=0.8})')
print(f' Cache hit rate : {cache_fin.hit_rate:.1%}')
print(f' p99 latency    : {np.percentile(lats_arr,99):.3f}ms')
print(f' Retrain events : {len(retrain_days)} in 90 days')
print()
print(' ALL 20-DAY SKILLS DEMONSTRATED:')
skills_20 = [
    'Day 1-5 : NumPy, Pandas, sklearn, XGBoost, FastAPI basics',
    'Day 6   : Bayesian stats, backprop, custom transformers',
    'Day 7   : Recommenders, A/B testing, churn pipeline',
    'Day 8   : Graph ML, autoencoder, fraud, CI/CD registry',
    'Day 9   : Embeddings, RAG v1, MLOps dashboard',
    'Day 10  : Data quality, multi-label, SHAP, online learning',
    'Day 11  : Causal inference, survival analysis, bandits',
    'Day 12  : Federated learning, differential privacy, active learning',
    'Day 13  : Optimizers, randomised SVD, PDP/ICE, bias audit',
    'Day 14  : Bayesian HPO, entity embeddings, multi-task',
    'Day 15  : Self-attention, Prophet forecasting, streaming',
    'Day 16  : MoE, SimCLR contrastive, conformal prediction',
    'Day 17  : Calibration, NAS, normalising flows, fairness constraints',
    'Day 18  : RLHF, Bradley-Terry, PPO, LoRA, DPO',
    'Day 19  : Vector databases, re-ranking, tool use, RAG v2',
    'Day 20  : Async batching, drift monitoring, cost optimisation, capstone',
]
for s in skills_20:
    print(f'   + {s}')
print('='*65)
print(' You are a production-ready ML Engineer.')
print('='*65)

In [ ]:
# Final Grand Visualisation
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. HPO convergence
best_so_far = [max([t.value for t in study_fin.trials[:i+1]]) for i in range(25)]
axes[0,0].plot(range(1,26), best_so_far, 'o-', color='#534AB7', linewidth=2, markersize=4)
axes[0,0].set_title('Bayesian HPO Convergence'); axes[0,0].set_xlabel('Trial')
axes[0,0].set_ylabel('Best AUC'); axes[0,0].grid(alpha=0.3)

# 2. Reliability diagram
frac_v, mp_v = calibration_curve(y_te_f, prob_te_cal, n_bins=8)
axes[0,1].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0,1].plot(mp_v, frac_v, 'o-', color='#1D9E75', linewidth=2, markersize=7)
axes[0,1].fill_between(mp_v, frac_v, mp_v, alpha=0.15, color='#1D9E75')
axes[0,1].set_title(f'Calibration (ECE={ece_after:.4f})')
axes[0,1].set_xlabel('Mean predicted prob'); axes[0,1].set_ylabel('Fraction positive')
axes[0,1].grid(alpha=0.3)

# 3. Throughput vs latency
axes[0,2].semilogx(batch_sizes, throughputs, 's-', color='#D85A30', linewidth=2, markersize=7)
axes[0,2].set_title('Throughput vs Batch Size'); axes[0,2].set_xlabel('Batch size')
axes[0,2].set_ylabel('Throughput (samples/sec)'); axes[0,2].grid(alpha=0.3)

# 4. Drift monitoring
axes[1,0].plot(df_mon['day'], df_mon['auc'], color='#534AB7', linewidth=1.5)
for d in df_mon[df_mon['retrain']]['day']:
    axes[1,0].axvline(d, color='#1D9E75', alpha=0.5, linewidth=0.8)
axes[1,0].set_title('90-Day AUC Monitoring (green=retrain)')
axes[1,0].set_xlabel('Day'); axes[1,0].set_ylabel('AUC'); axes[1,0].grid(alpha=0.3)

# 5. Fairness bar
pos_rates = [pred_fin[g_te_f==g].mean() for g in [0,1]]
axes[1,1].bar(['Group 0','Group 1'], pos_rates, color=['#534AB7','#1D9E75'], alpha=0.85)
axes[1,1].axhline(max(pos_rates)*0.8, linestyle='--', color='#D85A30', label='80% DI threshold')
axes[1,1].set_title(f'Fairness Audit (DI={di:.3f})')
axes[1,1].set_ylabel('Positive rate'); axes[1,1].legend(fontsize=8)

# 6. Cost-quality pareto
axes[1,2].scatter(df_cost['cost'], df_cost['accuracy'], alpha=0.3, color='#888780', s=30)
axes[1,2].plot(pareto_df['cost'], pareto_df['accuracy'], 'o-', color='#534AB7', linewidth=2, markersize=7)
axes[1,2].set_title('Cost-Quality Pareto Frontier')
axes[1,2].set_xlabel('Inference cost'); axes[1,2].set_ylabel('Accuracy')
axes[1,2].grid(alpha=0.3)

plt.suptitle('20-Day Grand Finale: Complete Production ML System', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Day 20 Summary

| Concept | Skill Gained |
|---|---|
| Async batch server | Queue + flush on max_batch or timeout |
| Priority queue serving | Heap-based priority with tier routing |
| Throughput vs latency | Batch size trade-off curve measurement |
| Wasserstein drift | Earth mover distance between distributions |
| MMD | Kernel-based two-sample test for drift |
| Concept drift | Label relationship change across time windows |
| Feature-level drift | Per-feature Wasserstein with alert thresholds |
| Semantic cache | MD5 hash cache with LRU eviction and cost report |
| Quantisation cost | float32/16/int8/int4 latency and cost comparison |
| Cost-quality Pareto | Find cheapest model meeting accuracy SLA |
| 20-day grand finale | All skills: HPO → calibrate → fairness → conformal → serve → monitor |

---
## 20-Day AI/ML Internship — COMPLETE

**You have now built 20 production ML systems from scratch.**

Skills span: data engineering, classical ML, deep learning, LLMs, RAG, RL, federated learning,
differential privacy, causal inference, active learning, model serving, drift monitoring,
cost optimisation, calibration, fairness, explainability, and research-level methods.

**Next steps:** Open-source contributions, Kaggle competitions, ML research papers, MLOps at scale.

In [ ]:
# 20-Day Internship in 20 Lines
print('20-DAY AI/ML INTERNSHIP — BONUS FINAL CELL')

# 1. Data + HPO
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Bayesian HPO
def obj(t):
    m = GradientBoostingClassifier(n_estimators=t.suggest_int('n',30,100),
                                    learning_rate=t.suggest_float('lr',0.01,0.3,log=True),
                                    random_state=42)
    return cross_val_score(m, Xtr, ytr, cv=3, scoring='roc_auc').mean()
st2 = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=0))
st2.optimize(obj, n_trials=15, show_progress_bar=False)

# 3. Train + Calibrate
best = GradientBoostingClassifier(**{k:v for k,v in st2.best_params.items()}, random_state=42).fit(Xtr, ytr)
Xcal, Xev, ycal, yev = train_test_split(Xte, yte, test_size=0.5, random_state=0)
iso2 = IsotonicRegression(out_of_bounds='clip').fit(best.predict_proba(Xcal)[:,1], ycal)
prob = iso2.predict(best.predict_proba(Xev)[:,1])

# 4. Report
print(f'  HPO AUC          : {st2.best_value:.4f}')
print(f'  Calibrated AUC   : {roc_auc_score(yev, prob):.4f}')
print(f'  ECE              : {ece_metric(yev, prob):.4f}')
print(f'  Cache hit rate   : {cache_fin.hit_rate:.1%}')
print(f'  Drift events     : {len(retrain_days)} in 90 days')
print()
print('20-DAY INTERNSHIP COMPLETE. You are production-ready!')